In [1]:
import math
import requests
import itertools
import folium
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm

sns.set(
    { "figure.figsize": (17, 7) },
    style='ticks',
    palette=sns.color_palette("Set2"),
    color_codes=True,
    font_scale=5
)

plt.rcParams.update({
    "axes.labelsize": 12,  # Axes label font size
})

%config InlineBackend.figure_format = 'retina'
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load data
stops_df = pd.read_csv("data/timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\stops.txt")
stop_times_df = pd.read_csv("data/timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\stop_times.txt")
trips = pd.read_csv("data/timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\trips.txt")

# Convert datatypes
stop_times_df["arrival_time"] = pd.to_timedelta(stop_times_df["arrival_time"])
stop_times_df["departure_time"] = pd.to_timedelta(stop_times_df["departure_time"])

# Merge route id details into stop times
stop_times_df = stop_times_df.merge(
    trips[["route_id", "trip_id"]],
    left_on='trip_id', 
    right_on='trip_id', 
    how='left')

# Isolate Bradford
stops_df = stops_df.loc[(stops_df["stop_lat"] < 54) & (stops_df["stop_lat"] > 53.7)].reset_index(drop=True)
stops_df = stops_df.loc[(stops_df["stop_lon"] < -1.55) & (stops_df["stop_lon"] > -1.95)].reset_index(drop=True)

# Get routes through each stop

In [3]:
stop_route_dict = stop_times_df.groupby("stop_id")["route_id"].unique().to_dict()
route_stops_dict = stop_times_df.groupby("route_id")["stop_id"].unique().to_dict()

# Find route v2

In [4]:
origin = (53.79, -1.73)
dest = (53.841401, -1.827540)

## Find route

In [5]:
lon_scale = math.cos(math.radians(origin[0]))

In [6]:
origin_dist_dict = dict(zip(
    stops_df["stop_id"], 
    np.sqrt((stops_df["stop_lat"] - origin[0])**2 + ((stops_df["stop_lon"] - origin[1]) * lon_scale)**2)
))

dest_dist_dict = dict(zip(
    stops_df["stop_id"], 
    np.sqrt((stops_df["stop_lat"] - dest[0])**2 + ((stops_df["stop_lon"] - dest[1]) * lon_scale)**2)
))

records = []
unique_routes = trips["route_id"].unique()

for route in unique_routes:
    stops = route_stops_dict.get(route, [])
    
    valid_origin = ((stop, origin_dist_dict[stop]) for stop in stops if stop in origin_dist_dict)
    valid_dest = ((stop, dest_dist_dict[stop]) for stop in stops if stop in dest_dist_dict)
    
    try:
        o_stop, o_dist = min(valid_origin, key=lambda x: x[1])
        d_stop, d_dist = min(valid_dest, key=lambda x: x[1])
        
        records.append({
            "route": route, 
            "origin near stop": o_stop, 
            "origin stop dist": o_dist,
            "dest near stop": d_stop,
            "dest stop dist": d_dist,
            "lowest dist sum": o_dist + d_dist 
        })
    except ValueError:
        continue

result_df = pd.DataFrame(records)
if not result_df.empty:
    result_df = result_df.sort_values(by="lowest dist sum").reset_index(drop=True)

result_df

,route,origin near stop,origin stop dist,dest near stop,dest stop dist,lowest dist sum
0,120468,450032484,0.011060,450021141,0.002420,0.013480
1,22584,450032484,0.011060,450021141,0.002420,0.013480
2,12767,450032483,0.011638,450021141,0.002420,0.014058
3,12777,450030020,0.013030,450021141,0.002420,0.015450
4,12779,450030020,0.013030,450021163,0.004158,0.017188
...,...,...,...,...,...,...
311,92967,3200YND77540,0.212415,3200YND81210,0.209774,0.422189
312,113362,3200YND77540,0.212415,3200YND81210,0.209774,0.422189
313,112572,3200YNA96443,0.219962,3200YNA96443,0.205463,0.425425
314,20835,3200YND81350,0.223514,3200YND81350,0.218975,0.442490


In [ ]:
for selected_route in range(len(result_df)):
    try:
        best_route_id = result_df.loc[selected_route, "route"]
        initial_stop = result_df.loc[selected_route, "origin near stop"]
        end_stop = result_df.loc[selected_route, "dest near stop"]

        route_df = stop_times_df[stop_times_df["route_id"] == best_route_id].copy()

        pivot = route_df[route_df["stop_id"].isin([initial_stop, end_stop])].pivot(
            index="trip_id", columns="stop_id", values="stop_sequence"
        )

        # Both stops must exist AND start must come before end
        valid_trips = pivot.dropna(subset=[initial_stop, end_stop])
        # valid_trips = valid_trips[valid_trips[initial_stop] < valid_trips[end_stop]]

        if valid_trips.empty:
            raise ValueError("No trips found where start stop occurs before end stop.")

        # Identify the longest trip among valid ones
        # Instead of another groupby, we can just look at the max sequence of the valid trip IDs
        longest_trip_id = route_df[route_df["trip_id"].isin(valid_trips.index)] \
                            .groupby("trip_id")["stop_sequence"].max().idxmax()

        # --- 2. Optimization: Prep Journey Data ---
        sample_journey = stop_times_df[stop_times_df["trip_id"] == longest_trip_id].copy()
        sample_journey = sample_journey.merge(stops_df[['stop_id', 'stop_lat', 'stop_lon']], on='stop_id')

        start_seq = valid_trips.loc[longest_trip_id, initial_stop]
        end_seq = valid_trips.loc[longest_trip_id, end_stop]
        if start_seq > end_seq: start_seq, end_seq = end_seq, start_seq

        # Filter to only the segment being travelled
        mask = (sample_journey["stop_sequence"] >= start_seq) & (sample_journey["stop_sequence"] <= end_seq)
        journey_segment = sample_journey[mask].sort_values("stop_sequence")

        break
    except:
        print(f"Found {selected_route + 1} routes which do not have complete trips to display.")
        continue

In [8]:
m = folium.Map(location=[53.79, -1.75], zoom_start=12, tiles='CartoDB positron')

# Draw the actual path as a PolyLine (Better visualization than just dots)
path_coords = journey_segment[['stop_lat', 'stop_lon']].values.tolist()
folium.PolyLine(path_coords, color="blue", weight=3, opacity=0.7).add_to(m)

# Add Start/End Markers
for point, color, label in [(origin, "green", "Origin"), (dest, "red", "Destination")]:
    folium.CircleMarker(location=point, radius=5, color=color, fill=True, popup=label).add_to(m)

# Efficiently add stop markers using itertuples (faster than iterrows or loc)
for stop in journey_segment.itertuples():
    is_key_stop = stop.stop_id in [initial_stop, end_stop]
    folium.CircleMarker(
        location=[stop.stop_lat, stop.stop_lon],
        radius=4 if is_key_stop else 2,
        color="blue" if is_key_stop else "gray",
        fill=True,
        fill_opacity=0.6,
        popup=f"Seq: {stop.stop_sequence}"
    ).add_to(m)

m.save("optimized_route_map.html")